# Entraînement CamemBERT v2

Notebook d'entraînement en deux étapes :

1. classification binaire `IA / non-IA` ;
2. classification multi-étiquette des compétences IA uniquement sur les formations IA confirmées.

Le notebook consomme `data/processed/dataset_entrainement.csv` et s'appuie sur le pipeline de nettoyage généré par `scripts/clean_and_merge_datasets.py`.


In [2]:
from __future__ import annotations

from pathlib import Path
import sys
import json
import math

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from datasets import Dataset
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    TrainingArguments,
    Trainer,
    set_seed,
)
from torch import nn


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'data' / 'processed' / 'dataset_entrainement.csv').exists():
            return candidate
    return start


ROOT = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(ROOT))

from scripts.clean_and_merge_datasets import build_text_modele, clean_text, normalize_unicode, parse_multi_values

set_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('ROOT =', ROOT)
print('DEVICE =', DEVICE)


/home/bibi/deepforma/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ROOT = /home/bibi/deepforma
DEVICE = cuda


## 1. Chargement des données

Le fichier `dataset_entrainement.csv` contient uniquement les exemples IA confirmés et non-IA confirmés.


In [3]:
DATA_PATH = ROOT / 'data' / 'processed' / 'dataset_entrainement.csv'
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Fichier introuvable : {DATA_PATH}')

df = pd.read_csv(DATA_PATH)
print('Dimensions :', df.shape)
print('Répartition :')
print(df['statut_annotation'].value_counts(dropna=False).to_string())

display(df.head(3))


Dimensions : (1235, 28)
Répartition :
statut_annotation
non_ia_confirmee    901
ia_confirmee        334


,formation_id,intitule,description,objectifs,programme,public_cible,prerequis,niveau,modalite,duree,...,est_lie_ia,statut_annotation,source_file,source_row,formation_group_id,cleaning_flags,certification_type,certification_code,source_tags,near_duplicate_cluster_id
0,9686f60b79a76a32,BTS Communication,NaN,NaN,NaN,Apprenants Bac+2 intégrant l'IA dans un cursus...,NaN,"BAC+2 : DEUG, BT, DUT... (NIVEAU 5)",À distance,2 an,...,1,ia_confirmee,Dataset_V7_Anton_CSV - Dataset_V7_Anton_CSV.cs...,1,9bb03b403e90fd0a,NaN,RNCP,37198.0,NaN,dup_00401
1,d2b08161c2944845,BTS Communication,NaN,NaN,NaN,Apprenants Bac+2 intégrant l'IA dans un cursus...,NaN,"BAC+2 : DEUG, BT, DUT... (NIVEAU 5)",À distance,2 an,...,1,ia_confirmee,Dataset_V7_Anton_CSV - Dataset_V7_Anton_CSV.cs...,2,9bb03b403e90fd0a,NaN,RNCP,37198.0,NaN,dup_00401
2,6a51bdeddecc7b33,Améliorer l'efficacité de sa TPE à l'aide de l'IA,NaN,NaN,NaN,Dirigeants de TPE/PME souhaitant intégrer l'IA...,NaN,0,À distance,35 h,...,1,ia_confirmee,Dataset_V7_Anton_CSV - Dataset_V7_Anton_CSV.cs...,3,388424a2594693a6,NaN,RS,7311.0,NaN,dup_00094


## 2. Préparation du split sans fuite

Les groupes reposent sur `formation_group_id` afin de garder ensemble les doublons exacts et les variantes proches.


In [4]:
def grouped_train_val_test_split(frame: pd.DataFrame, group_col: str = 'formation_group_id', seed: int = 42):
    if group_col not in frame.columns:
        raise KeyError(f'Colonne manquante : {group_col}')

    groups = frame[group_col].astype(str)
    splitter = GroupShuffleSplit(n_splits=1, train_size=0.70, random_state=seed)
    train_idx, temp_idx = next(splitter.split(frame, groups=groups))
    train_df = frame.iloc[train_idx].copy()
    temp_df = frame.iloc[temp_idx].copy()

    temp_groups = temp_df[group_col].astype(str)
    splitter2 = GroupShuffleSplit(n_splits=1, train_size=0.50, random_state=seed)
    val_idx, test_idx = next(splitter2.split(temp_df, groups=temp_groups))
    val_df = temp_df.iloc[val_idx].copy()
    test_df = temp_df.iloc[test_idx].copy()

    return train_df, val_df, test_df


train_df, val_df, test_df = grouped_train_val_test_split(df)
for name, subset in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(f'{name}: lignes={len(subset)} groupes={subset["formation_group_id"].nunique()}')

assert set(train_df['formation_group_id']).isdisjoint(set(val_df['formation_group_id']))
assert set(train_df['formation_group_id']).isdisjoint(set(test_df['formation_group_id']))
assert set(val_df['formation_group_id']).isdisjoint(set(test_df['formation_group_id']))


train: lignes=857 groupes=825
val: lignes=190 groupes=177
test: lignes=188 groupes=177


## 3. Analyse du déséquilibre

On calcule les distributions de classes pour le modèle binaire et les occurrences de compétences pour le modèle multi-étiquette.


In [5]:
binary_counts = train_df['statut_annotation'].value_counts()
print(binary_counts.to_string())

ia_train = train_df[train_df['statut_annotation'] == 'ia_confirmee'].copy()
comp_counter = {}
for value in ia_train['competences_ia'].fillna(''):
    for comp in parse_multi_values(value):
        comp_counter[comp] = comp_counter.get(comp, 0) + 1

comp_stats = pd.DataFrame(sorted(comp_counter.items(), key=lambda x: (-x[1], x[0])), columns=['competence', 'occurrences'])
print('Compétences les plus fréquentes :')
display(comp_stats.head(20))
print('Compétences rares (< 20 occurrences) :')
display(comp_stats[comp_stats['occurrences'] < 20])


statut_annotation
non_ia_confirmee    625
ia_confirmee        232
Compétences les plus fréquentes :


,competence,occurrences
0,Éthique de l’IA,166
1,IA générative,148
2,Prompt Engineering,114
3,Automatisation,93
4,NLP,79
5,Machine Learning,61
6,Visualisation,59
7,Data Science,57
8,Data Engineering,56
9,Deep Learning,53


Compétences rares (< 20 occurrences) :


,competence,occurrences
15,Reinforcement Learning,11
16,Computer Vision,9
17,Gestion de projet IA,7


## 4. Modèle 1 - classification IA / non-IA

On affine `camembert-base` avec une tête de classification binaire et une perte pondérée pour gérer le déséquilibre.


In [6]:
MODEL_NAME = 'camembert-base'
TEXT_COL = 'texte_modele'
LABEL_COL = 'binary_label'
MAX_LENGTH = 256

label_map = {'non_ia_confirmee': 0, 'ia_confirmee': 1}
train_bin = train_df.copy()
val_bin = val_df.copy()
test_bin = test_df.copy()
for frame in [train_bin, val_bin, test_bin]:
    frame[LABEL_COL] = frame['statut_annotation'].map(label_map).astype(int)

classes = np.array([0, 1])
class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=train_bin[LABEL_COL].to_numpy())
class_weights = torch.tensor(class_weights, dtype=torch.float32)
print('Class weights :', class_weights.tolist())

tokenizer_bin = AutoTokenizer.from_pretrained(MODEL_NAME)
model_bin = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)


Class weights : [0.6855999827384949, 1.846982717514038]


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 2848.84it/s]
[transformers] CamembertForSequenceClassification LOAD REPORT from: camembert-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [20]:
import inspect

import numpy as np
import torch
import torch.nn as nn

from datasets import Dataset
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
)
from transformers import (
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)


# ============================================================
# 1. Vérifications
# ============================================================

required_variables = {
    "train_bin": train_bin,
    "val_bin": val_bin,
    "test_bin": test_bin,
    "tokenizer_bin": tokenizer_bin,
    "model_bin": model_bin,
    "class_weights": class_weights,
    "TEXT_COL": TEXT_COL,
    "LABEL_COL": LABEL_COL,
    "MAX_LENGTH": MAX_LENGTH,
    "ROOT": ROOT,
}

for variable_name, variable_value in required_variables.items():
    if variable_value is None:
        raise ValueError(
            f"La variable `{variable_name}` est absente ou vaut None."
        )


dataframes = {
    "train_bin": train_bin,
    "val_bin": val_bin,
    "test_bin": test_bin,
}

for dataset_name, dataframe in dataframes.items():
    if dataframe.empty:
        raise ValueError(
            f"Le DataFrame `{dataset_name}` est vide."
        )

    missing_columns = {
        TEXT_COL,
        LABEL_COL,
    } - set(dataframe.columns)

    if missing_columns:
        raise KeyError(
            f"Colonnes absentes dans `{dataset_name}` : "
            f"{sorted(missing_columns)}"
        )


if getattr(model_bin.config, "num_labels", None) != 2:
    raise ValueError(
        "Le modèle binaire doit avoir `num_labels=2`. "
        f"Valeur actuelle : {model_bin.config.num_labels}"
    )


class_weights = torch.as_tensor(
    class_weights,
    dtype=torch.float32,
)

if class_weights.numel() != 2:
    raise ValueError(
        "Deux poids de classes sont attendus : "
        "un pour le label 0 et un pour le label 1. "
        f"Poids reçus : {class_weights.tolist()}"
    )

print("Vérifications terminées.")
print("Poids de classes :", class_weights.tolist())


# ============================================================
# 2. Préparation des DataFrames
# ============================================================

def prepare_binary_dataframe(dataframe):
    prepared_df = dataframe[
        [TEXT_COL, LABEL_COL]
    ].copy()

    prepared_df = prepared_df.dropna(
        subset=[TEXT_COL, LABEL_COL]
    )

    prepared_df[TEXT_COL] = (
        prepared_df[TEXT_COL]
        .astype(str)
        .str.strip()
    )

    prepared_df = prepared_df.loc[
        prepared_df[TEXT_COL].ne("")
    ].copy()

    prepared_df[LABEL_COL] = (
        prepared_df[LABEL_COL]
        .astype(int)
    )

    invalid_labels = (
        set(prepared_df[LABEL_COL].unique()) - {0, 1}
    )

    if invalid_labels:
        raise ValueError(
            "Les labels doivent uniquement être 0 ou 1. "
            f"Labels invalides : {sorted(invalid_labels)}"
        )

    return prepared_df.reset_index(drop=True)


train_bin_prepared = prepare_binary_dataframe(train_bin)
val_bin_prepared = prepare_binary_dataframe(val_bin)
test_bin_prepared = prepare_binary_dataframe(test_bin)

print(
    f"Train : {len(train_bin_prepared)} lignes\n"
    f"Validation : {len(val_bin_prepared)} lignes\n"
    f"Test : {len(test_bin_prepared)} lignes"
)


# ============================================================
# 3. Conversion Hugging Face Dataset
# ============================================================

train_bin_ds = Dataset.from_pandas(
    train_bin_prepared,
    preserve_index=False,
)

val_bin_ds = Dataset.from_pandas(
    val_bin_prepared,
    preserve_index=False,
)

test_bin_ds = Dataset.from_pandas(
    test_bin_prepared,
    preserve_index=False,
)


# ============================================================
# 4. Tokenisation
# ============================================================

def tokenize_binary(batch):
    return tokenizer_bin(
        batch[TEXT_COL],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )


train_bin_ds = train_bin_ds.map(
    tokenize_binary,
    batched=True,
    desc="Tokenisation train",
)

val_bin_ds = val_bin_ds.map(
    tokenize_binary,
    batched=True,
    desc="Tokenisation validation",
)

test_bin_ds = test_bin_ds.map(
    tokenize_binary,
    batched=True,
    desc="Tokenisation test",
)


def finalize_binary_dataset(dataset):
    dataset = dataset.rename_column(
        LABEL_COL,
        "labels",
    )

    columns_to_remove = [
        column
        for column in [TEXT_COL]
        if column in dataset.column_names
    ]

    if columns_to_remove:
        dataset = dataset.remove_columns(
            columns_to_remove
        )

    return dataset


train_bin_ds = finalize_binary_dataset(train_bin_ds)
val_bin_ds = finalize_binary_dataset(val_bin_ds)
test_bin_ds = finalize_binary_dataset(test_bin_ds)

collator_bin = DataCollatorWithPadding(
    tokenizer=tokenizer_bin,
    return_tensors="pt",
)


# ============================================================
# 5. Trainer pondéré
# ============================================================

class WeightedBinaryTrainer(Trainer):
    def __init__(self, class_weights=None, **kwargs):
        super().__init__(**kwargs)

        if class_weights is None:
            class_weights = torch.ones(
                2,
                dtype=torch.float32,
            )

        self.class_weights = torch.as_tensor(
            class_weights,
            dtype=torch.float32,
        )

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
    ):
        model_inputs = dict(inputs)

        labels = model_inputs.pop(
            "labels"
        ).long()

        outputs = model(**model_inputs)
        logits = outputs.logits

        if logits.ndim != 2 or logits.shape[-1] != 2:
            raise ValueError(
                "Le modèle doit produire des logits de forme "
                f"[batch_size, 2]. Forme reçue : {tuple(logits.shape)}"
            )

        weights = self.class_weights.to(
            device=logits.device,
            dtype=logits.dtype,
        )

        loss_function = nn.CrossEntropyLoss(
            weight=weights
        )

        loss = loss_function(
            logits,
            labels,
        )

        return (
            (loss, outputs)
            if return_outputs
            else loss
        )


# ============================================================
# 6. Métriques
# ============================================================

def compute_binary_metrics(eval_pred):
    predictions = eval_pred.predictions
    labels = eval_pred.label_ids

    if isinstance(predictions, tuple):
        predictions = predictions[0]

    predicted_labels = np.argmax(
        predictions,
        axis=-1,
    )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            labels,
            predicted_labels,
            average="binary",
            pos_label=1,
            zero_division=0,
        )
    )

    macro_precision, macro_recall, macro_f1, _ = (
        precision_recall_fscore_support(
            labels,
            predicted_labels,
            average="macro",
            zero_division=0,
        )
    )

    return {
        "accuracy": accuracy_score(
            labels,
            predicted_labels,
        ),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
    }


# ============================================================
# 7. Arguments d'entraînement
# ============================================================

training_arguments_parameters = inspect.signature(
    TrainingArguments.__init__
).parameters

evaluation_argument = {}

if "eval_strategy" in training_arguments_parameters:
    evaluation_argument["eval_strategy"] = "epoch"
else:
    evaluation_argument["evaluation_strategy"] = "epoch"


use_cuda = torch.cuda.is_available()

use_bf16 = (
    use_cuda
    and hasattr(torch.cuda, "is_bf16_supported")
    and torch.cuda.is_bf16_supported()
)

binary_args = TrainingArguments(
    output_dir=str(
        ROOT / "models" / "binary_ia_v2"
    ),
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    fp16=use_cuda and not use_bf16,
    bf16=use_bf16,
    seed=42,
    data_seed=42,
    **evaluation_argument,
)


# ============================================================
# 8. Construction du Trainer
# ============================================================

trainer_parameters = {
    "model": model_bin,
    "args": binary_args,
    "train_dataset": train_bin_ds,
    "eval_dataset": val_bin_ds,
    "data_collator": collator_bin,
    "compute_metrics": compute_binary_metrics,
    "class_weights": class_weights,
    "callbacks": [
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ],
}

trainer_signature = inspect.signature(
    Trainer.__init__
).parameters

if "processing_class" in trainer_signature:
    trainer_parameters["processing_class"] = tokenizer_bin
else:
    trainer_parameters["tokenizer"] = tokenizer_bin


binary_trainer = WeightedBinaryTrainer(
    **trainer_parameters
)

print("Trainer binaire prêt.")

Vérifications terminées.
Poids de classes : [0.6855999827384949, 1.846982717514038]
Train : 857 lignes
Validation : 190 lignes
Test : 188 lignes


Tokenisation test: 100%|██████████| 188/188 [00:00<00:00, 20635.10 examples/s]


Trainer binaire prêt.


In [9]:
def evaluate_binary(trainer, dataset, frame, title):
    metrics = trainer.predict(dataset)
    preds = np.argmax(metrics.predictions, axis=-1)
    labels = frame[LABEL_COL].to_numpy()
    cm = confusion_matrix(labels, preds)
    print(title)
    print('Accuracy :', accuracy_score(labels, preds))
    print('F1 :', f1_score(labels, preds, zero_division=0))
    print('Confusion matrix :', cm)
    return preds, cm

# Example after training:
preds_val, cm_val = evaluate_binary(binary_trainer, val_bin_ds, val_bin, 'Validation binaire')


Validation binaire
Accuracy : 0.49473684210526314
F1 : 0.38461538461538464
Confusion matrix : [[64 74]
 [22 30]]


## 5. Modèle 2 - multi-étiquette des compétences IA

Le second modèle est entraîné uniquement sur les formations IA confirmées. Les cibles sont binarisées avec `MultiLabelBinarizer`.


In [10]:
ia_train = train_df[train_df['statut_annotation'] == 'ia_confirmee'].copy()
ia_val = val_df[val_df['statut_annotation'] == 'ia_confirmee'].copy()
ia_test = test_df[test_df['statut_annotation'] == 'ia_confirmee'].copy()

if ia_train.empty:
    raise ValueError('Aucune formation IA confirmée dans le train split.')

train_labels = [parse_multi_values(v) for v in ia_train['competences_ia'].fillna('')]
val_labels = [parse_multi_values(v) for v in ia_val['competences_ia'].fillna('')]
test_labels = [parse_multi_values(v) for v in ia_test['competences_ia'].fillna('')]

mlb = MultiLabelBinarizer()
Y_train = mlb.fit_transform(train_labels)
Y_val = mlb.transform(val_labels) if len(val_labels) else np.zeros((0, len(mlb.classes_)), dtype=int)
Y_test = mlb.transform(test_labels) if len(test_labels) else np.zeros((0, len(mlb.classes_)), dtype=int)
print('Nombre de compétences :', len(mlb.classes_))
print('Compétences :', list(mlb.classes_))

model_ml = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(mlb.classes_),
    problem_type='multi_label_classification',
)
tokenizer_ml = AutoTokenizer.from_pretrained(MODEL_NAME)


Nombre de compétences : 18
Compétences : ['Automatisation', 'Big Data', 'Computer Vision', 'Data Engineering', 'Data Science', 'Deep Learning', 'Gestion de projet IA', 'IA générative', 'Machine Learning', 'NLP', 'No-code / Low-code', 'Prompt Engineering', 'Python pour l’IA', 'RAG', 'Reinforcement Learning', 'Séries temporelles', 'Visualisation', 'Éthique de l’IA']


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 16093.29it/s]
[transformers] CamembertForSequenceClassification LOAD REPORT from: camembert-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
import inspect

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from datasets import Dataset
from sklearn.metrics import (
    f1_score,
    precision_recall_fscore_support,
)
from transformers import (
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)


# ============================================================
# 1. Vérifications
# ============================================================

NUM_LABELS = Y_train.shape[1]

if len(ia_train) != len(Y_train):
    raise ValueError(
        f"Décalage train : {len(ia_train)} textes "
        f"contre {len(Y_train)} vecteurs de labels."
    )

if len(ia_val) != len(Y_val):
    raise ValueError(
        f"Décalage validation : {len(ia_val)} textes "
        f"contre {len(Y_val)} vecteurs de labels."
    )

if len(ia_test) != len(Y_test):
    raise ValueError(
        f"Décalage test : {len(ia_test)} textes "
        f"contre {len(Y_test)} vecteurs de labels."
    )

if Y_train.ndim != 2:
    raise ValueError(
        f"Y_train doit être une matrice 2D. "
        f"Forme reçue : {Y_train.shape}"
    )

if getattr(model_ml.config, "num_labels", None) != NUM_LABELS:
    raise ValueError(
        "Le nombre de sorties du modèle ne correspond pas "
        "au nombre de compétences.\n"
        f"model_ml.config.num_labels = {model_ml.config.num_labels}\n"
        f"Y_train.shape[1] = {NUM_LABELS}"
    )

print(f"Nombre de compétences : {NUM_LABELS}")
print(f"Train IA : {len(ia_train)}")
print(f"Validation IA : {len(ia_val)}")
print(f"Test IA : {len(ia_test)}")


# ============================================================
# 2. Construction des DataFrames
# ============================================================

def create_multilabel_dataframe(frame, labels):
    """
    Crée un DataFrame texte + vecteur multi-étiquette.
    """

    if frame is None or len(frame) == 0:
        return None

    if len(frame) != len(labels):
        raise ValueError(
            f"Nombre de textes différent du nombre de labels : "
            f"{len(frame)} contre {len(labels)}."
        )

    texts = (
        frame[TEXT_COL]
        .fillna("")
        .astype(str)
        .str.strip()
        .tolist()
    )

    labels_array = np.asarray(
        labels,
        dtype=np.float32,
    )

    valid_indices = [
        index
        for index, text in enumerate(texts)
        if text
    ]

    if not valid_indices:
        return None

    clean_texts = [
        texts[index]
        for index in valid_indices
    ]

    clean_labels = labels_array[
        valid_indices
    ]

    return pd.DataFrame(
        {
            TEXT_COL: clean_texts,
            "labels": clean_labels.tolist(),
        }
    )


train_ml_df = create_multilabel_dataframe(
    ia_train,
    Y_train,
)

val_ml_df = create_multilabel_dataframe(
    ia_val,
    Y_val,
)

test_ml_df = create_multilabel_dataframe(
    ia_test,
    Y_test,
)

if train_ml_df is None or train_ml_df.empty:
    raise ValueError(
        "Le jeu d'entraînement multi-étiquette est vide."
    )


# ============================================================
# 3. Conversion en Dataset Hugging Face
# ============================================================

train_ml = Dataset.from_pandas(
    train_ml_df,
    preserve_index=False,
)

val_ml = (
    Dataset.from_pandas(
        val_ml_df,
        preserve_index=False,
    )
    if val_ml_df is not None
    else None
)

test_ml = (
    Dataset.from_pandas(
        test_ml_df,
        preserve_index=False,
    )
    if test_ml_df is not None
    else None
)


# ============================================================
# 4. Tokenisation
# ============================================================

def tokenize_multilabel(batch):
    return tokenizer_ml(
        batch[TEXT_COL],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )


train_ml = train_ml.map(
    tokenize_multilabel,
    batched=True,
    desc="Tokenisation multi-label train",
)

if val_ml is not None:
    val_ml = val_ml.map(
        tokenize_multilabel,
        batched=True,
        desc="Tokenisation multi-label validation",
    )

if test_ml is not None:
    test_ml = test_ml.map(
        tokenize_multilabel,
        batched=True,
        desc="Tokenisation multi-label test",
    )


def remove_text_column(dataset):
    if dataset is None:
        return None

    if TEXT_COL in dataset.column_names:
        dataset = dataset.remove_columns(
            [TEXT_COL]
        )

    return dataset


train_ml = remove_text_column(train_ml)
val_ml = remove_text_column(val_ml)
test_ml = remove_text_column(test_ml)


# ============================================================
# 5. Calcul des poids positifs
# ============================================================

train_label_counts = np.asarray(
    Y_train,
    dtype=np.float32,
).sum(axis=0)

negative_counts = (
    len(Y_train) - train_label_counts
)

raw_pos_weight = (
    negative_counts
    / np.clip(
        train_label_counts,
        1.0,
        None,
    )
)

# Évite qu'une compétence très rare domine entièrement la loss.
MAX_POS_WEIGHT = 20.0

clipped_pos_weight = np.clip(
    raw_pos_weight,
    1.0,
    MAX_POS_WEIGHT,
)

pos_weight = torch.tensor(
    clipped_pos_weight,
    dtype=torch.float32,
)

print("\nOccurrences par compétence :")
print(train_label_counts.astype(int).tolist())

print("\nPoids positifs utilisés :")
print(pos_weight.tolist())


# ============================================================
# 6. Trainer pondéré multi-étiquette
# ============================================================

class WeightedMultiLabelTrainer(Trainer):
    def __init__(self, pos_weight=None, **kwargs):
        super().__init__(**kwargs)

        if pos_weight is None:
            pos_weight = torch.ones(
                NUM_LABELS,
                dtype=torch.float32,
            )

        self.pos_weight = torch.as_tensor(
            pos_weight,
            dtype=torch.float32,
        )

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
    ):
        # Ne pas modifier directement le dictionnaire reçu.
        model_inputs = dict(inputs)

        labels = model_inputs.pop(
            "labels"
        ).float()

        outputs = model(**model_inputs)
        logits = outputs.logits

        if logits.ndim != 2:
            raise ValueError(
                "Les logits doivent être une matrice 2D "
                f"[batch_size, num_labels]. Reçu : {logits.shape}"
            )

        if logits.shape[-1] != NUM_LABELS:
            raise ValueError(
                f"Le modèle produit {logits.shape[-1]} sorties, "
                f"mais {NUM_LABELS} compétences sont attendues."
            )

        weights = self.pos_weight.to(
            device=logits.device,
            dtype=logits.dtype,
        )

        loss_function = nn.BCEWithLogitsLoss(
            pos_weight=weights,
        )

        loss = loss_function(
            logits,
            labels,
        )

        return (
            (loss, outputs)
            if return_outputs
            else loss
        )


# ============================================================
# 7. Métriques multi-étiquette
# ============================================================

MULTILABEL_THRESHOLD = 0.5


def sigmoid_numpy(values):
    """
    Sigmoïde numériquement stable.
    """
    values = np.clip(
        values,
        -50,
        50,
    )

    return 1.0 / (
        1.0 + np.exp(-values)
    )


def compute_multilabel_metrics(eval_pred):
    logits = eval_pred.predictions
    labels = eval_pred.label_ids

    if isinstance(logits, tuple):
        logits = logits[0]

    labels = np.asarray(
        labels,
        dtype=np.int32,
    )

    probabilities = sigmoid_numpy(logits)

    preds = (
        probabilities >= MULTILABEL_THRESHOLD
    ).astype(np.int32)

    precision_micro, recall_micro, f1_micro, _ = (
        precision_recall_fscore_support(
            labels,
            preds,
            average="micro",
            zero_division=0,
        )
    )

    precision_macro, recall_macro, f1_macro, _ = (
        precision_recall_fscore_support(
            labels,
            preds,
            average="macro",
            zero_division=0,
        )
    )

    precision_samples, recall_samples, f1_samples, _ = (
        precision_recall_fscore_support(
            labels,
            preds,
            average="samples",
            zero_division=0,
        )
    )

    exact_match = np.mean(
        np.all(
            labels == preds,
            axis=1,
        )
    )

    return {
        "f1_micro": f1_micro,
        "f1_macro": f1_macro,
        "f1_samples": f1_samples,
        "precision_micro": precision_micro,
        "recall_micro": recall_micro,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "precision_samples": precision_samples,
        "recall_samples": recall_samples,
        "exact_match": exact_match,
    }


# ============================================================
# 8. Compatibilité Transformers
# ============================================================

training_arguments_parameters = inspect.signature(
    TrainingArguments.__init__
).parameters

evaluation_argument = {}

if "eval_strategy" in training_arguments_parameters:
    evaluation_argument["eval_strategy"] = "epoch"
else:
    evaluation_argument["evaluation_strategy"] = "epoch"


use_cuda = torch.cuda.is_available()

use_bf16 = (
    use_cuda
    and hasattr(torch.cuda, "is_bf16_supported")
    and torch.cuda.is_bf16_supported()
)


# Early stopping et load_best_model nécessitent un jeu validation.
has_validation = (
    val_ml is not None
    and len(val_ml) > 0
)

training_arguments_values = {
    "output_dir": str(
        ROOT
        / "models"
        / "multilabel_competences_v2"
    ),
    "learning_rate": 2e-5,
    "per_device_train_batch_size": 8,
    "per_device_eval_batch_size": 8,
    "num_train_epochs": 4,
    "weight_decay": 0.01,
    "logging_strategy": "steps",
    "logging_steps": 20,
    "save_total_limit": 2,
    "report_to": "none",
    "fp16": use_cuda and not use_bf16,
    "bf16": use_bf16,
    "seed": 42,
    "data_seed": 42,
}

if has_validation:
    training_arguments_values.update(
        {
            "save_strategy": "epoch",
            "load_best_model_at_end": True,
            "metric_for_best_model": "f1_micro",
            "greater_is_better": True,
            **evaluation_argument,
        }
    )
else:
    training_arguments_values.update(
        {
            "save_strategy": "no",
            "load_best_model_at_end": False,
        }
    )

multi_args = TrainingArguments(
    **training_arguments_values
)


# ============================================================
# 9. Construction du Trainer
# ============================================================

trainer_parameters = {
    "model": model_ml,
    "args": multi_args,
    "train_dataset": train_ml,
    "eval_dataset": val_ml,
    "data_collator": DataCollatorWithPadding(
        tokenizer=tokenizer_ml,
        return_tensors="pt",
    ),
    "compute_metrics": (
        compute_multilabel_metrics
        if has_validation
        else None
    ),
    "pos_weight": pos_weight,
}

if has_validation:
    trainer_parameters["callbacks"] = [
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ]

trainer_signature = inspect.signature(
    Trainer.__init__
).parameters

if "processing_class" in trainer_signature:
    trainer_parameters[
        "processing_class"
    ] = tokenizer_ml
else:
    trainer_parameters[
        "tokenizer"
    ] = tokenizer_ml


multi_trainer = WeightedMultiLabelTrainer(
    **trainer_parameters
)

print("\nTrainer multi-étiquette prêt.")

Nombre de compétences : 18
Train IA : 232
Validation IA : 52
Test IA : 50


Tokenisation multi-label test: 100%|██████████| 50/50 [00:00<00:00, 14309.17 examples/s]



Occurrences par compétence :
[93, 26, 9, 56, 57, 53, 7, 148, 61, 79, 30, 114, 50, 45, 11, 23, 59, 166]

Poids positifs utilisés :
[1.4946236610412598, 7.92307710647583, 20.0, 3.142857074737549, 3.0701754093170166, 3.3773584365844727, 20.0, 1.0, 2.803278684616089, 1.9367088079452515, 6.733333110809326, 1.0350877046585083, 3.640000104904175, 4.155555725097656, 20.0, 9.086956977844238, 2.9322032928466797, 1.0]

Trainer multi-étiquette prêt.


In [12]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    classification_report,
    f1_score,
    precision_recall_fscore_support,
)


def stable_sigmoid(logits):
    """
    Calcule une sigmoïde stable numériquement.
    """
    logits = np.clip(logits, -50, 50)
    return 1.0 / (1.0 + np.exp(-logits))


def evaluate_multilabel(
    trainer,
    dataset,
    mlb,
    threshold=0.5,
    title="Évaluation multi-étiquette",
):
    """
    Évalue un modèle multi-étiquette à partir des labels stockés
    directement dans le Dataset Hugging Face.

    Cette fonction ne dépend pas d'une colonne `competences_ia`
    dans un DataFrame Pandas.
    """

    if dataset is None:
        raise ValueError("Le dataset multi-étiquette est absent.")

    if len(dataset) == 0:
        raise ValueError("Le dataset multi-étiquette est vide.")

    if "labels" not in dataset.column_names:
        raise KeyError(
            "Le Dataset Hugging Face ne contient pas de colonne `labels`."
        )

    prediction_output = trainer.predict(dataset)

    logits = prediction_output.predictions

    if isinstance(logits, tuple):
        logits = logits[0]

    probabilities = stable_sigmoid(logits)

    predictions = (
        probabilities >= threshold
    ).astype(np.int32)

    # Utiliser en priorité les labels renvoyés par Trainer
    true_labels = prediction_output.label_ids

    if true_labels is None:
        true_labels = np.asarray(
            dataset["labels"],
            dtype=np.int32,
        )
    else:
        true_labels = np.asarray(
            true_labels,
            dtype=np.int32,
        )

    if predictions.shape != true_labels.shape:
        raise ValueError(
            "La forme des prédictions ne correspond pas à celle des labels : "
            f"{predictions.shape} contre {true_labels.shape}."
        )

    if predictions.shape[1] != len(mlb.classes_):
        raise ValueError(
            "Le nombre de sorties du modèle ne correspond pas au nombre "
            "de classes du MultiLabelBinarizer : "
            f"{predictions.shape[1]} sorties contre "
            f"{len(mlb.classes_)} classes."
        )

    precision_micro, recall_micro, f1_micro, _ = (
        precision_recall_fscore_support(
            true_labels,
            predictions,
            average="micro",
            zero_division=0,
        )
    )

    precision_macro, recall_macro, f1_macro, _ = (
        precision_recall_fscore_support(
            true_labels,
            predictions,
            average="macro",
            zero_division=0,
        )
    )

    precision_samples, recall_samples, f1_samples, _ = (
        precision_recall_fscore_support(
            true_labels,
            predictions,
            average="samples",
            zero_division=0,
        )
    )

    exact_match = np.mean(
        np.all(
            true_labels == predictions,
            axis=1,
        )
    )

    predicted_count_per_sample = predictions.sum(axis=1)
    true_count_per_sample = true_labels.sum(axis=1)

    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)

    print(f"Seuil                 : {threshold:.3f}")
    print(f"Nombre d'exemples     : {len(true_labels)}")
    print(f"Nombre de compétences : {len(mlb.classes_)}")

    print("\nMétriques globales :")
    print(f"Précision micro : {precision_micro:.4f}")
    print(f"Rappel micro    : {recall_micro:.4f}")
    print(f"F1 micro        : {f1_micro:.4f}")

    print(f"\nPrécision macro : {precision_macro:.4f}")
    print(f"Rappel macro    : {recall_macro:.4f}")
    print(f"F1 macro        : {f1_macro:.4f}")

    print(f"\nF1 samples      : {f1_samples:.4f}")
    print(f"Exact match     : {exact_match:.4f}")

    print(
        "\nNombre moyen de compétences réelles par formation : "
        f"{true_count_per_sample.mean():.2f}"
    )

    print(
        "Nombre moyen de compétences prédites par formation : "
        f"{predicted_count_per_sample.mean():.2f}"
    )

    # Statistiques classe par classe
    class_precision, class_recall, class_f1, class_support = (
        precision_recall_fscore_support(
            true_labels,
            predictions,
            average=None,
            zero_division=0,
        )
    )

    per_class_metrics = pd.DataFrame(
        {
            "competence": mlb.classes_,
            "support": class_support.astype(int),
            "precision": class_precision,
            "recall": class_recall,
            "f1": class_f1,
            "predictions_positives": predictions.sum(axis=0),
        }
    )

    per_class_metrics = per_class_metrics.sort_values(
        by=["f1", "support"],
        ascending=[True, True],
    ).reset_index(drop=True)

    print("\nCompétences les plus difficiles :")
    display(per_class_metrics.head(15))

    print("\nRapport de classification :")
    print(
        classification_report(
            true_labels,
            predictions,
            target_names=[
                str(label)
                for label in mlb.classes_
            ],
            zero_division=0,
        )
    )

    true_competences = mlb.inverse_transform(
        true_labels
    )

    predicted_competences = mlb.inverse_transform(
        predictions
    )

    detailed_predictions = pd.DataFrame(
        {
            "competences_reelles": [
                list(values)
                for values in true_competences
            ],
            "competences_predites": [
                list(values)
                for values in predicted_competences
            ],
            "nombre_reel": true_count_per_sample,
            "nombre_predit": predicted_count_per_sample,
        }
    )

    metrics = {
        "threshold": float(threshold),
        "precision_micro": float(precision_micro),
        "recall_micro": float(recall_micro),
        "f1_micro": float(f1_micro),
        "precision_macro": float(precision_macro),
        "recall_macro": float(recall_macro),
        "f1_macro": float(f1_macro),
        "precision_samples": float(precision_samples),
        "recall_samples": float(recall_samples),
        "f1_samples": float(f1_samples),
        "exact_match": float(exact_match),
        "mean_true_labels": float(
            true_count_per_sample.mean()
        ),
        "mean_predicted_labels": float(
            predicted_count_per_sample.mean()
        ),
    }

    return {
        "metrics": metrics,
        "predictions": predictions,
        "probabilities": probabilities,
        "true_labels": true_labels,
        "per_class_metrics": per_class_metrics,
        "detailed_predictions": detailed_predictions,
    }

## 6. Fonction de prédiction

La prédiction s'effectue en cascade :

- le modèle binaire calcule `probabilite_ia` ;
- si la probabilité est sous le seuil, le modèle de compétences n'est pas sollicité ;
- sinon le second modèle renvoie les compétences au-dessus du seuil.


In [13]:
IA_THRESHOLD = 0.50
COMPETENCE_THRESHOLD = 0.35

def build_model_input(formation_data) -> pd.Series:
    if isinstance(formation_data, pd.Series):
        row = formation_data.copy()
    elif isinstance(formation_data, dict):
        row = pd.Series(formation_data)
    else:
        raise TypeError('formation_data doit être un dict ou une Series pandas')

    if 'texte_modele' not in row or not clean_text(row.get('texte_modele', '')):
        base = {
            'intitule': row.get('intitule', row.get('Intitulé', row.get('Intitulé de la formation', ''))),
            'description': row.get('description', ''),
            'objectifs': row.get('objectifs', ''),
            'programme': row.get('programme', ''),
            'public_cible': row.get('public_cible', row.get('Public cible', '')),
            'prerequis': row.get('prerequis', ''),
            'niveau': row.get('niveau', row.get('Niveau', '')),
            'modalite': row.get('modalite', row.get('Modalité', '')),
            'duree': row.get('duree', row.get('Durée', '')),
            'certification': row.get('certification', row.get('Type de certification', '')),
            'codes_rome': row.get('codes_rome', row.get('Codes ROME', '')),
            'organisme': row.get('organisme', row.get('Organisme de formation', '')),
        }
        row = pd.Series(base)
        row['texte_modele'] = build_text_modele(row)
    return row


def predict_formation(formation_data):
    row = build_model_input(formation_data)
    text = clean_text(row.get('texte_modele', ''))
    binary_inputs = tokenizer_bin(text, return_tensors='pt', truncation=True, max_length=MAX_LENGTH).to(DEVICE)
    model_bin.eval()
    model_bin.to(DEVICE)
    with torch.no_grad():
        logits = model_bin(**binary_inputs).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]
    probabilite_ia = float(probs[1])
    result = {
        'est_lie_ia': probabilite_ia >= IA_THRESHOLD,
        'probabilite_ia': probabilite_ia,
        'competences': [],
    }
    if probabilite_ia < IA_THRESHOLD:
        return result

    model_ml.eval()
    model_ml.to(DEVICE)
    multi_inputs = tokenizer_ml(text, return_tensors='pt', truncation=True, max_length=MAX_LENGTH).to(DEVICE)
    with torch.no_grad():
        logits = model_ml(**multi_inputs).logits
        probabilities = torch.sigmoid(logits).cpu().numpy()[0]

    ranked = sorted(
        [
            {'nom': name, 'probabilite': float(prob)}
            for name, prob in zip(mlb.classes_, probabilities)
            if prob >= COMPETENCE_THRESHOLD
        ],
        key=lambda item: item['probabilite'],
        reverse=True,
    )
    result['competences'] = ranked
    return result

# Exemple d'usage :
predict_formation({'intitule': 'Formation IA générative pour les RH', 'description': '...', 'public_cible': '...'})


{'est_lie_ia': False, 'probabilite_ia': 0.4983977973461151, 'competences': []}

## 7. Évaluation finale et sauvegarde

Une fois l'entraînement terminé, évaluez sur le test split, sauvegardez les modèles et documentez les seuils retenus.


In [14]:
import json
from pathlib import Path

import numpy as np


# ============================================================
# 1. Vérifications
# ============================================================

if binary_trainer is None:
    raise ValueError("Le trainer binaire n'est pas disponible.")

if multi_trainer is None:
    raise ValueError("Le trainer multi-étiquette n'est pas disponible.")

if test_bin_ds is None or len(test_bin_ds) == 0:
    raise ValueError("Le jeu de test binaire est vide ou absent.")

if test_ml is None or len(test_ml) == 0:
    raise ValueError("Le jeu de test multi-étiquette est vide ou absent.")


# Utiliser les DataFrames effectivement employés pour créer les datasets
binary_test_frame = (
    test_bin_prepared
    if "test_bin_prepared" in globals()
    else test_bin
)

multilabel_test_frame = (
    test_ml_df
    if "test_ml_df" in globals() and test_ml_df is not None
    else ia_test
)


# ============================================================
# 2. Évaluation du modèle binaire
# ============================================================

print("\n" + "=" * 60)
print("ÉVALUATION DU MODÈLE BINAIRE")
print("=" * 60)

binary_metrics_test = binary_trainer.evaluate(
    eval_dataset=test_bin_ds,
    metric_key_prefix="test_binary",
)

print("\nMétriques retournées par Trainer :")
for metric_name, metric_value in binary_metrics_test.items():
    if isinstance(metric_value, (int, float, np.integer, np.floating)):
        print(f"{metric_name}: {float(metric_value):.4f}")
    else:
        print(f"{metric_name}: {metric_value}")


preds_binary_test, cm_binary_test = evaluate_binary(
    trainer=binary_trainer,
    dataset=test_bin_ds,
    frame=binary_test_frame,
    title="Test binaire",
)


# ============================================================
# 3. Évaluation du modèle multi-étiquette
# ============================================================

print("\n" + "=" * 60)
print("ÉVALUATION DU MODÈLE MULTI-ÉTIQUETTE")
print("=" * 60)

multilabel_metrics_test = multi_trainer.evaluate(
    eval_dataset=test_ml,
    metric_key_prefix="test_multilabel",
)

print("\nMétriques retournées par Trainer :")
for metric_name, metric_value in multilabel_metrics_test.items():
    if isinstance(metric_value, (int, float, np.integer, np.floating)):
        print(f"{metric_name}: {float(metric_value):.4f}")
    else:
        print(f"{metric_name}: {metric_value}")


multilabel_evaluation_result = evaluate_multilabel(
    trainer=multi_trainer,
    dataset=test_ml,
    mlb=mlb,
    threshold=COMPETENCE_THRESHOLD,
    title="Test multi-étiquette",
)

# ============================================================
# 4. Création des dossiers de sortie
# ============================================================

binary_final_dir = (
    ROOT
    / "models"
    / "binary_ia_v2"
    / "final"
)

multilabel_final_dir = (
    ROOT
    / "models"
    / "multilabel_competences_v2"
    / "final"
)

binary_final_dir.mkdir(
    parents=True,
    exist_ok=True,
)

multilabel_final_dir.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 5. Sauvegarde des modèles
# ============================================================

binary_trainer.save_model(
    str(binary_final_dir)
)

multi_trainer.save_model(
    str(multilabel_final_dir)
)


# Sauvegarde explicite des tokenizers
tokenizer_bin.save_pretrained(
    str(binary_final_dir)
)

tokenizer_ml.save_pretrained(
    str(multilabel_final_dir)
)


# ============================================================
# 6. Sauvegarde du MultiLabelBinarizer
# ============================================================

# Les classes doivent impérativement conserver le même ordre
# qu'au moment de l'entraînement.
multilabel_classes = [
    str(label)
    for label in mlb.classes_
]

with open(
    multilabel_final_dir / "label_classes.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        multilabel_classes,
        file,
        ensure_ascii=False,
        indent=2,
    )


# ============================================================
# 7. Sauvegarde des seuils
# ============================================================

threshold_config = {
    "binary_threshold": None,
    "multilabel_threshold": float(
        COMPETENCE_THRESHOLD
    ),
}

with open(
    multilabel_final_dir / "thresholds.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        threshold_config,
        file,
        ensure_ascii=False,
        indent=2,
    )


# ============================================================
# 8. Conversion des métriques en JSON
# ============================================================
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch


def make_json_serializable(value):
    """
    Convertit récursivement les objets courants du notebook
    vers des types compatibles avec json.dump().
    """

    # Valeurs nulles et types JSON natifs
    if value is None or isinstance(value, (str, bool, int, float)):
        return value

    # Dictionnaires
    if isinstance(value, dict):
        return {
            str(key): make_json_serializable(item)
            for key, item in value.items()
        }

    # Listes, tuples et ensembles
    if isinstance(value, (list, tuple, set)):
        return [
            make_json_serializable(item)
            for item in value
        ]

    # DataFrame Pandas :
    # chaque ligne devient un dictionnaire
    if isinstance(value, pd.DataFrame):
        return [
            make_json_serializable(record)
            for record in value.to_dict(orient="records")
        ]

    # Série Pandas
    if isinstance(value, pd.Series):
        return make_json_serializable(
            value.to_list()
        )

    # Index Pandas
    if isinstance(value, pd.Index):
        return make_json_serializable(
            value.to_list()
        )

    # Types NumPy
    if isinstance(value, np.ndarray):
        return make_json_serializable(
            value.tolist()
        )

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        numeric_value = float(value)

        # JSON standard ne gère pas proprement NaN et Infinity
        if not np.isfinite(numeric_value):
            return None

        return numeric_value

    if isinstance(value, np.bool_):
        return bool(value)

    # Valeurs Pandas/NumPy manquantes
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass

    # Tenseurs PyTorch
    if isinstance(value, torch.Tensor):
        tensor_value = value.detach().cpu()

        if tensor_value.ndim == 0:
            return make_json_serializable(
                tensor_value.item()
            )

        return make_json_serializable(
            tensor_value.tolist()
        )

    # Chemins pathlib
    if isinstance(value, Path):
        return str(value)

    # Dernier recours pour certains objets possédant to_dict()
    if hasattr(value, "to_dict"):
        try:
            return make_json_serializable(
                value.to_dict()
            )
        except Exception:
            pass

    # Éviter que la sauvegarde entière échoue sur un objet inconnu
    return str(value)


ÉVALUATION DU MODÈLE BINAIRE


[transformers] early stopping required metric_for_best_model, but did not find eval_f1 so early stopping is disabled


Training Loss,Validation Loss,Epoch,Binary Loss,Binary Accuracy,Binary Precision,Binary Recall,Binary F1,Binary Macro Precision,Binary Macro Recall,Binary Macro F1
No log,No log,0,0.693786,0.478723,0.289474,0.660000,0.402439,0.529872,0.536522,0.470087



Métriques retournées par Trainer :
test_binary_loss: 0.6938
test_binary_accuracy: 0.4787
test_binary_precision: 0.2895
test_binary_recall: 0.6600
test_binary_f1: 0.4024
test_binary_macro_precision: 0.5299
test_binary_macro_recall: 0.5365
test_binary_macro_f1: 0.4701


Test binaire
Accuracy : 0.4787234042553192
F1 : 0.4024390243902439
Confusion matrix : [[57 81]
 [17 33]]

ÉVALUATION DU MODÈLE MULTI-ÉTIQUETTE


...


Métriques retournées par Trainer :
test_multilabel_loss: 1.0447
test_multilabel_f1_micro: 0.3731
test_multilabel_f1_macro: 0.2815
test_multilabel_f1_samples: 0.3656
test_multilabel_precision_micro: 0.2544
test_multilabel_recall_micro: 0.6991
test_multilabel_precision_macro: 0.2084
test_multilabel_recall_macro: 0.6235
test_multilabel_precision_samples: 0.2524
test_multilabel_recall_samples: 0.7045
test_multilabel_exact_match: 0.0000



Test multi-étiquette
Seuil                 : 0.350
Nombre d'exemples     : 50
Nombre de compétences : 18

Métriques globales :
Précision micro : 0.2511
Rappel micro    : 1.0000
F1 micro        : 0.4014

Précision macro : 0.2511
Rappel macro    : 0.9444
F1 macro        : 0.3733

F1 samples      : 0.3949
Exact match     : 0.0000

Nombre moyen de compétences réelles par formation : 4.52
Nombre moyen de compétences prédites par formation : 18.00

Compétences les plus difficiles :


,competence,support,precision,recall,f1,predictions_positives
0,Séries temporelles,0,0.00,0.0,0.000000,50
1,Reinforcement Learning,1,0.02,1.0,0.039216,50
2,Gestion de projet IA,2,0.04,1.0,0.076923,50
3,Computer Vision,3,0.06,1.0,0.113208,50
4,No-code / Low-code,5,0.10,1.0,0.181818,50
5,Big Data,8,0.16,1.0,0.275862,50
6,Automatisation,10,0.20,1.0,0.333333,50
7,Python pour l’IA,11,0.22,1.0,0.360656,50
8,RAG,13,0.26,1.0,0.412698,50
9,Data Engineering,14,0.28,1.0,0.437500,50



Rapport de classification :
                        precision    recall  f1-score   support

        Automatisation       0.20      1.00      0.33        10
              Big Data       0.16      1.00      0.28         8
       Computer Vision       0.06      1.00      0.11         3
      Data Engineering       0.28      1.00      0.44        14
          Data Science       0.34      1.00      0.51        17
         Deep Learning       0.30      1.00      0.46        15
  Gestion de projet IA       0.04      1.00      0.08         2
         IA générative       0.52      1.00      0.68        26
      Machine Learning       0.38      1.00      0.55        19
                   NLP       0.30      1.00      0.46        15
    No-code / Low-code       0.10      1.00      0.18         5
    Prompt Engineering       0.44      1.00      0.61        22
      Python pour l’IA       0.22      1.00      0.36        11
                   RAG       0.26      1.00      0.41        13
Reinforcem

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.20it/s]
